In [ ]:
from pathlib import Path 
import re
import numpy as np
import pandas as pd

In [ ]:
# Task 1: File ingestion and regular expressions

def read_file(file_path):
    path = Path(file_path)
    if not path.exists():
        raise FileNotFoundError(f"File not found: {path}")
    
    # Try to read file in utf-8 encoding, if it fails, try iso-8859-1 encoding
    try:
        return path.read_text(encoding="utf-8"), "utf-8"
    except UnicodeDecodeError:
        return path.read_text(encoding="iso-8859-1"), "iso-8859-1"


In [ ]:
# Regex that matches a valid sales line and captures the six fields
# (timestamp, register_id, item, price, currency, status). Noise lines do not match.
LOG_PATTERN = re.compile(
    r"^\[(?P<timestamp>\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2})\]\s+"
    r"\[(?P<register_id>[A-Za-z0-9_]+)\]\s+"
    r"item=(?P<item>.+?)\s+"
    r"price=(?P<price>\d+(?:\.\d+)?)\s+"
    r"(?P<currency>[A-Z]{3})\s+"
    r"\((?P<status>SUCCESS|REFUND|CANCELLED)\)\s*$"
)

def parse_log(text):
    records = []
    for line in text.splitlines():
        match = LOG_PATTERN.match(line.strip())
        if match:
            record = match.groupdict()
            record["price"] = float(record["price"])
            records.append(record)
    return records

In [ ]:
# Task 2: Recursive utility functions

# Recursively find all .log files in a directory and its subfolders.
def find_all_logfiles(directory):
    path = Path(directory)
    if not path.is_dir():
        raise NotADirectoryError(f"Not a directory: {path}")
    
    logfiles = []
    for item in path.iterdir():
        if item.is_file() and item.suffix == ".log":
            logfiles.append(item)
        elif item.is_dir():
            logfiles.extend(find_all_logfiles(item))

        return sorted(logfiles)



In [ ]:
# Recursively find all register IDs marked as active in nested dictionaries. 
def find_active_registers(data):
    active_registers = set()


    if isinstance(data, dict):
        if data.get("active") is True and "register_id" in data:
            active_registers.add(data["register_id"])
        for value in data.values():
            active_registers.update(find_active_registers(value))

    elif isinstance(data, list):
        for item in data:
            active_registers.update(find_active_registers(item))

    return active_registers



In [ ]:
# Task 3: Functions for cleaning and analysing data using NumPy

# Replace prices more than 3 standard deviations from the mean with the median of the valid prices
def remove_price_outliers(prices):
    prices = np.asarray(prices, dtype=float).ravel()  # 1D float array
    if prices.size == 0:
        return prices

    mean_price = prices.mean()
    std_dev = prices.std()

    outliers = np.abs(prices - mean_price) > 3 * std_dev  # True = outlier
    median_valid = np.median(prices[~outliers])           # median of valid prices

    cleaned = prices.copy()
    cleaned[outliers] = median_valid
    return cleaned




In [ ]:
# Compute the element-wise refund ratio, refund / (sales + 1e-6)
def compute_refund_ratio(sales_count: np.ndarray,
                         refund_count: np.ndarray) -> np.ndarray:
    sales_count = np.asarray(sales_count, dtype=float)
    refund_count = np.asarray(refund_count, dtype=float)
    return refund_count / (sales_count + 1e-6)



In [ ]:
# Task 4: Data analysis using Pandas

# Convert transaction dicts to a DataFrame indexed and sorted by timestamp
def transactions_to_dataframe(records):
    df = pd.DataFrame(records)
    df["timestamp"] = pd.to_datetime(df["timestamp"], format="%Y-%m-%d %H:%M:%S")
    df = df.set_index("timestamp")
    return df.sort_index()




In [ ]:
# Group by hour and compute total sales and refunds
def hourly_sales_by_item(df):
    return df.groupby("item").resample("1h")["price"].sum()



In [ ]:
def revenue_pivot_by_status(df):
    pivot = pd.pivot_table(df, index="item", columns="status", values="price", aggfunc="sum", fill_value=0)